# 05 Learning Curves

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`lc7.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/lc7.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)
- [`lcl2.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/lcl2.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/HGB/Normal/scripts/lc7.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'a7c579a7c37c8bb8f65afdedf377a80a6bb9def6'}, '05_model_optimize/HGB/Normal/scripts/lcl2.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'befe8be0dff49b90372e93acf9c4f29a2f1d6066'}}

## lc7

원본: [05_model_optimize/HGB/Normal/scripts/lc7.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/lc7.py)  
Git blob: `a7c579a7c37c8bb8f65afdedf377a80a6bb9def6`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/lc7.py
from pathlib import Path
import sys,time,json,numpy as np,pandas as pd
sys.argv=['x','none']; exec(open(Path(__file__).resolve().parent/'run_all.py').read().split("stage=sys.argv[1]")[0])
Ftr=np.load(OUT/'Ftr.npy'); Fva=np.load(OUT/'Fva.npy'); Fm={int(k):v for k,v in np.load(OUT/'Fm.npz').items()}; Fhm=np.vstack([Fm[w] for w in W])
keep=np.r_[0:784,840:1164]; A,B,C=Ftr[:,keep],Fva[:,keep],Fhm[:,keep]
m=hgb(); t=time.time(); m.fit(A,ytr); fs=time.time()-t
t=time.time(); pv=m.predict(B); pvs=time.time()-t; t=time.time(); ph=m.predict(C); phs=time.time()-t
t=time.time(); m.predict(B[:1]); one=time.time()-t
it=np.arange(len(m.train_score_))
av=[accuracy_score(yva,p) for p in m.staged_predict(B)]; ah=[accuracy_score(yhm,p) for p in m.staged_predict(C)]
pd.DataFrame({'iteration':it,'train_loss':-m.train_score_,'val_loss':-m.validation_score_,'official_val_acc':[np.nan]+av[:len(it)-1],'handmade_acc':[np.nan]+ah[:len(it)-1]}).to_csv(OUT/'lc7.csv',index=False)
json.dump(dict(fit_s=round(fs),iters=int(m.n_iter_),pred_val_s=round(pvs,3),pred_hm_s=round(phs,3),ms_per_img=round(1000*pvs/len(yva),4),single_ms=round(1000*one,2),hm=accuracy_score(yhm,ph),val=accuracy_score(yva,pv)),open(OUT/'lc7.json','w'))
print('ok',fs,m.n_iter_,flush=True)


## lcl2

원본: [05_model_optimize/HGB/Normal/scripts/lcl2.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/lcl2.py)  
Git blob: `befe8be0dff49b90372e93acf9c4f29a2f1d6066`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/lcl2.py
from pathlib import Path
import sys,time,json,numpy as np,pandas as pd
kind=sys.argv[1]; sys.argv=['x','none']; exec(open(Path(__file__).resolve().parent/'run_all.py').read().split("stage=sys.argv[1]")[0])
from sklearn.ensemble import HistGradientBoostingClassifier
Ftr=np.load(OUT/'Ftr.npy'); Fva=np.load(OUT/'Fva.npy'); Fm={int(k):v for k,v in np.load(OUT/'Fm.npz').items()}; keep=np.r_[0:784,840:1164]
m=HistGradientBoostingClassifier(max_iter=300,early_stopping=True,validation_fraction=0.1,n_iter_no_change=10,l2_regularization=1.0,random_state=42)
if kind=='n9':
    X,y=Ftr[:,keep],ytr; E=np.vstack([Fm[w] for w in W])[:,keep]; ye=yhm
else:
    o=[41,42,43]; X=np.vstack([Ftr[:,keep]]+[Fm[x][:,keep] for x in o]); y=np.concatenate([ytr]+[wr[x]['y'] for x in o]); E=Fm[44][:,keep]; ye=wr[44]['y']
t=time.time(); m.fit(X,y); fs=time.time()-t
B=Fva[:,keep]; t=time.time(); pv=m.predict(B); pvs=time.time()-t; t=time.time(); pe=m.predict(E); pes=time.time()-t
it=np.arange(len(m.train_score_))
av=[accuracy_score(yva,p) for p in m.staged_predict(B)]; ae=[accuracy_score(ye,p) for p in m.staged_predict(E)]
mx=[max(float(np.abs(t_.nodes['value'][t_.nodes['is_leaf']==1]).max()) for t_ in i_) for i_ in m._predictors]
pd.DataFrame({'iteration':it,'train_loss':-m.train_score_,'val_loss':-m.validation_score_,'official_val_acc':[np.nan]+av[:len(it)-1],'eval_acc':[np.nan]+ae[:len(it)-1],'max_leaf':[np.nan]+mx[:len(it)-1]}).to_csv(OUT/f'lc_{kind}.csv',index=False)
json.dump(dict(fit_s=round(fs),iters=int(m.n_iter_),val=accuracy_score(yva,pv),eval=accuracy_score(ye,pe),pred_val_s=round(pvs,3),pred_eval_s=round(pes,3),ms_per_img=round(1000*pvs/len(yva),4)),open(OUT/f'lc_{kind}.json','w')); print('ok',flush=True)


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")